In [1]:
"""
Ovarian CT Classification - Malignancy & Subtype Prediction
===========================================================
Architecture: ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)
Strategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)

구조 변경:
 - Model A: Malignancy 분류 (기존 구조 유지)
 - Model B: Subtype 분류 (완전 분리, malignant-only 데이터)
 - 학습 분리: Step1(malignancy) -> Step2(subtype) -> Step3(cascade 평가)
 - 평균 pooling 사용
 - cascade_inference: inference 시에만 cascade 적용
"""

'\nOvarian CT Classification - Malignancy & Subtype Prediction\n===========================================================\nArchitecture: ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)\nStrategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)\n\n구조 변경:\n - Model A: Malignancy 분류 (기존 구조 유지)\n - Model B: Subtype 분류 (완전 분리, malignant-only 데이터)\n - 학습 분리: Step1(malignancy) -> Step2(subtype) -> Step3(cascade 평가)\n - 평균 pooling 사용\n - cascade_inference: inference 시에만 cascade 적용\n'

In [2]:
import os
import random
import numpy as np
from collections import OrderedDict, defaultdict, Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ViT_B_16_Weights
import torchvision.models as tv_models
from torchvision import transforms
import torchvision.transforms.functional as TF
from pathlib import Path
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                             f1_score, recall_score, confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

In [3]:
# 0. Config
DATA_ROOT = Path("/tf/nasw/dataset001/preprocessed/npz_us_augmented")
OUTPUT_ROOT = Path("./Cascade_models")

PRETRAINED = {
    "rad_resnet18": Path("/tf/pretrained_model/RadImageNet_resnet18.pth"),
    "rad_resnet50": Path("/tf/pretrained_model/RadImageNet_resnet50.pth"),
    "resnet18": Path("/tf/pretrained_model/resnet18-f37072fd.pth"),
    "resnet50": Path("/tf/pretrained_model/resnet50-11ad3fa6.pth"),
    "vit": Path("/tf/pretrained_model/vit_b_16-c867db91.pth"),
}

FOLDS = [1, 2, 3, 4, 5]
NUM_FOLDS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4


# Malignancy 모델 hyperparameter
MAL_BATCH_SIZE = 8
MAL_EPOCHS = 30
MAL_LR_BACKBONE = 1e-5
MAL_LR_HEAD = 5e-5
MAL_WEIGHT_DECAY = 1e-4
MAL_PATIENCE = 12
MAL_MIN_DELTA = 0.001
THRESHOLD = 0.5

# Malignancy 모델 hyperparameter
SUB_BATCH_SIZE = 8
SUB_EPOCHS = 80
SUB_LR_BACKBONE = 5e-6
SUB_LR_HEAD = 2e-5
SUB_WEIGHT_DECAY = 1e-4
SUB_PATIENCE = 20
SUB_MIN_DELTA = 0.001
FOCAL_GAMMA = 1.0
LABEL_SMOOTHING = 0.1

MALIGNANT_MAP: dict[int, int] = {}
NUM_MALIGNANT_SUBTYPES: int = 0

def infer_malignant_type_ids(data_root, folds):
    label_set_by_type = defaultdict(set)

    for fold_idx in FOLDS:
        for split_name in ["train", "val"]:
            npz_path = Path(data_root) / f"fold_{fold_idx}_{split_name}.npz"
            data = np.load(npz_path, allow_pickle=False)

            labels = data["labels"].astype(int)
            tumor_types = data["tumor_types"].astype(int)

            for y, t in zip(labels, tumor_types):
                label_set_by_type[int(t)].add(int(y))
                
        conflicts = {
            t: sorted(list(v))
            for t, v in label_set_by_type.items()
            if len(v) > 1
        }

        if len(conflicts) > 0:
            raise ValueError(
                f"tumor_type가 양성/악성 양쪽에 섞여있음: {conflicts}\n"
            )
        malignant_type_ids = sorted([
            t for t, labs in label_set_by_type.items()
            if labs == {1}
        ])

        print("malignant_type_ids:", malignant_type_ids)
        return malignant_type_ids


MALIGNANT_TYPE_IDS = infer_malignant_type_ids(DATA_ROOT, FOLDS)
MALIGNANT_MAP = {tid: i for i, tid in enumerate(MALIGNANT_TYPE_IDS)}
NUM_MALIGNANT_SUBTYPES = len(MALIGNANT_TYPE_IDS)
print("NUM_MALIGNANT_SUBTYPES:", NUM_MALIGNANT_SUBTYPES)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

malignant_type_ids: [0, 1, 3, 5, 7]
NUM_MALIGNANT_SUBTYPES: 5


In [4]:
# 1. Dataset 정의
# online augmentation 추가

class CTAugment:
    def __call__(self, x):
        # Gaussian noise
        if random.random() < 0.5:
            x = x + torch.randn_like(x) * 0.05

        # Random erasing
        if random.random() < 0.3:
            x = transforms.RandomErasing(
                p=1.0, scale=(0.02, 0.08), ratio=(0.3, 3.3)
            )(x)
        return x
    
class OvarianCTNPZDataset(Dataset):
    """
    Loads pre-processed CT slices from a .npz file,
    Expected keys: 'images', 'labels', 'tumor_types', 'patients'
    Malignancy 모델 학습 및 cascade 평가에 사용
    """
    
    def __init__(self, npz_path: Path, augment: bool = False):
        super().__init__()
        self.npz = np.load(npz_path, allow_pickle=False)

        self.images = self.npz["images"]
        self.labels = self.npz["labels"].astype(np.float32)
        self.tumor_types = self.npz["tumor_types"].astype(np.int64)
        self.patient_ids = (self.npz["patient_ids"] if "patient_ids" in self.npz.files 
                            else np.arange(len(self.labels)))

        # RadImageNet normalization constants
        self.mean = torch.tensor([0.204, 0.204, 0.204], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.286, 0.286, 0.286], dtype=torch.float32).view(1, 3, 1, 1)
        
        # Online augmentation (train only)
        self.augment = augment
        self.geo_aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            CTAugment(),
        ]) if augment else None

        self.color_aug = transforms.Compose([
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]) if augment else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        tumor_type = int(self.tumor_types[idx])


        # (S, H, W, C) -> (S, C, H, W), normalize to [0,1] then ImageNet-nomalize
        x = torch.from_numpy(x).float().permute(0, 3, 1, 2).contiguous()
        x = x / 255.0

        # ColorJitter는 [0, 1] 범위에서 적용
        if self.color_aug is not None:
            x = torch.stack([self.color_aug(x[s]) for s in range(x.shape[0])])

        
        x = (x-self.mean) / self.std

        if self.geo_aug is not None:
            x = torch.stack([self.geo_aug(x[s]) for s in range(x.shape[0])])

        if y == 1:
            if tumor_type not in MALIGNANT_MAP:
                raise ValueError(
                    f"malignant sample인데 tumor_type={tumor_type}가 MAP에 없음"
                )
            mal_subtype = MALIGNANT_MAP[tumor_type]
            mal_mask = True
        else:
            mal_subtype = -1
            mal_mask = False
            
        return {
            "image": x,
            "label": torch.tensor(y, dtype = torch.float32),
            "mal_subtype": torch.tensor(mal_subtype, dtype=torch.long),
            "mal_mask": torch.tensor(mal_mask, dtype=torch.bool),
        }


class MalignantOnlyDataset(Dataset):
    """
    악성 샘플만 필터링한 데이터셋
    Subtype 모델 학습에 사용
    """
    def __init__(self, npz_path: Path, augment: bool = False):
        base = OvarianCTNPZDataset(npz_path, augment=augment)
        self.base = base
        self.indices = [i for i in range(len(base)) if base.labels[i] == 1]

        counts = Counter()
        for i in self.indices:
            t = int(base.tumor_types[i])
            counts[MALIGNANT_MAP[t]] += 1
        print(f"  [MalignantOnly] 총 {len(self.indices)}명 |"
              f"subtype 분포: {[counts[k] for k in range(NUM_MALIGNANT_SUBTYPES)]}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

    def get_sample_weights(self) -> list[float]:
        """ WeightRandomSampler용 샘플 가중치 계산 """
        counts = Counter()
        for i in self.indices:
            t = int(self.base.tumor_types[i])
            counts[MALIGNANT_MAP[t]] += 1

        weights = []
        for i in self.indices:
            t = int(self.base.tumor_types[i])
            sub = MALIGNANT_MAP[t]
            weights.append(1.0 / counts[sub])

        return weights

In [5]:
# 2. Encoders
def _load_local_weights(model: nn.Module, path: Path, strict: bool = True) -> nn.Module:
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state, strict=strict)
    return model

def _load_rad_imagenet_weights(model: nn.Module, path: Path) -> nn.Module:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict):
        if "model" in ckpt:
            state = ckpt["model"]
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
        else:
            state = ckpt

    else:
        raise ValueError(f"Unexpected checkpoint type: {type(ckpt)}")

    # _orig_mod. prefix 제거
    state = {k.replace("_orig_mod.", ""): v 
             for k, v in state.items()
            }

    # fc layer 제외
    state = {k: v for k, v in state.items() if not k.startswith("fc.")}

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  [RadImageNet] loaded {path.name}")
    if missing:
        print(f"   missing keys ({len(missing)}): {missing[:3]}{'...' if len(missing) > 3 else ''}")
    if unexpected:
        print(f"   unexpected keys ({len(unexpected)}): {unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")

    return model
    
def build_encoder(freeze_backbone: bool = True, 
                  unfreeze_layer4: bool = True,
                  unfreeze_layer3: bool = False) -> tuple[nn.Module, int]:
    
    """
    RadImageNet ResNet18 encoder
    """
    encoder = tv_models.resnet18(weights=None)
    encoder = _load_rad_imagenet_weights(encoder, PRETRAINED["rad_resnet18"])
    feat_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False
        if unfreeze_layer4:
            for p in encoder.layer4.parameters():
                p.requires_grad = True
        if unfreeze_layer3:
            for p in encoder.layer3.parameters():
                p.requires_grad = True
    
    return encoder, feat_dim

In [6]:
# 3. Classifers

class MalignancyModel(nn.Module):
    """
    Malignancy 이진 분류 모델
    Attention pooling -> shared head -> binary output.
    """
    def __init__(self, encoder: nn.Module, feat_dim: int, 
                 hidden_dim: int = 256,
                 dropout: float = 0.3):
        super().__init__()
        self.encoder = encoder
        
        # Attention pooling: feat_dim -> feat_dim//2 -> 1
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.Tanh(),
            nn.Linear(feat_dim // 2, 1)
        )

        self.head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape

        # Encode all slices in parallel
        feat = self.encoder(x.view(B * S, C, H, W)).view(B, S, -1) # (B, S, feat_dim)
        
        # Attention-weighted patient-level pooling
        attn_weight = torch.softmax(self.attn(feat), dim=1) # (B, S, 1)
        pooled = (feat * attn_weight).sum(dim=1) # (B, feate_dim)
        
        return self.head(pooled).squeeze(1)



class SubtypeModel(nn.Module):
    """
    Subtype 다중 분류 모델
    단순 평균 pooling -> head.
    """
    def __init__(self, encoder: nn.Module, feat_dim: int, 
                 num_subtypes: int = NUM_MALIGNANT_SUBTYPES,
                 hidden_dim: int = 256, dropout: float = 0.5):
        super().__init__()
        self.encoder = encoder
    

        self.head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_subtypes)
        )

    def forward(self, x):
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape

        # Encode all slices in parallel
        feat = self.encoder(x.view(B * S, C, H, W)).view(B, S, -1) # (B, S, feat_dim)
        pooled = feat.mean(dim=1) # (B, feate_dim)
        
        return self.head(pooled)

In [7]:
# 4. Loss

class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss.
        FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)
    - alpha (class_weight): 다수 클래스 억제 (ENS weight 그대로 사용)
    - gamma: easy example 억제 (기본값 2.0)
    - smoothing: 정답 타겟을 1.0 -> (1-smoothing)으로 낮춰 과적합 억제
    """
    def __init__(self, weight: torch.Tensor, gamma: float = 2.0,
                 smoothing: float = 0.0):
        super().__init__()
        self.register_buffer("weight", weight.float())
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(1)
        log_prob = nn.functional.log_softmax(logits, dim=1) # (N, C)
        prob = log_prob.exp() # (N, C)

        # alpha: 각 샘플의 클래스 weight
        alpha = self.weight[targets] # (N,)

        # p_t: 정답 클래스의 확률
        p_t = prob[torch.arange(len(targets)), targets] # (N,)

        # Focal weight
        focal_w = (1.0 - p_t) ** self.gamma # (N,)

        if self.smoothing > 0.0:
            # soft target: 정답=1-s, 나머지=s/(C-1)
            smooth_val = self.smoothing / (num_classes - 1)
            soft_target = torch.full_like(log_prob, smooth_val)
            soft_target.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            loss_per_sample = -(soft_target * log_prob).sum(dim=1)
        else:
            loss_per_sample = -log_prob[torch.arange(len(targets)), targets]

        loss = (alpha * focal_w * loss_per_sample).mean()
        return loss

In [8]:
# 5. DataLoader factory (per fold)
def make_malignancy_loaders(fold_idx: int) -> tuple:
    train_npz = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz = DATA_ROOT / f"fold_{fold_idx}_val.npz"
    assert train_npz.exists(), f"Missing: {train_npz}"
    assert val_npz.exists(), f"Missing: {val_npz}"

    train_dataset = OvarianCTNPZDataset(train_npz, augment=True)
    val_dataset = OvarianCTNPZDataset(val_npz, augment=False)

    train_loader = DataLoader(train_dataset, batch_size=MAL_BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS, pin_memory = True)
    val_loader = DataLoader(val_dataset, batch_size=MAL_BATCH_SIZE, 
                            shuffle = False,num_workers = NUM_WORKERS, pin_memory = True)

    neg = (train_dataset.labels == 0).sum()
    pos = (train_dataset.labels == 1).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE,
                              dtype=torch.float32)

    print(f"[FOLD {fold_idx}] "
          f"benign(0)={(train_dataset.labels==0).sum()} "
          f"malignant(1)={(train_dataset.labels==1).sum()} "
          f"pos_weight={pos_weight.item():.4f}")

    return train_loader, val_loader, pos_weight

def make_subtype_loaders(fold_idx: int) -> tuple:
    train_npz = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz = DATA_ROOT / f"fold_{fold_idx}_val.npz"
    assert train_npz.exists(), f"Missing: {train_npz}"
    assert val_npz.exists(), f"Missing: {val_npz}"

    train_dataset = MalignantOnlyDataset(train_npz, augment=True)
    val_dataset = MalignantOnlyDataset(val_npz, augment=False)

    
    train_loader = DataLoader(train_dataset, batch_size=SUB_BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS, pin_memory = True)
    val_loader = DataLoader(val_dataset, batch_size=SUB_BATCH_SIZE, 
                            shuffle = False,num_workers = NUM_WORKERS, pin_memory = True)

    counts = np.array([
        sum(1 for i in train_dataset.indices
            if MALIGNANT_MAP[int(train_dataset.base.tumor_types[i])] == c)
        for c in range(NUM_MALIGNANT_SUBTYPES)
    ], dtype=np.float64)
    counts = np.maximum(counts, 1.0)
    beta = 0.9999
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    weights_ens = 1.0 / effective_num
    weights_ens = weights_ens / weights_ens.min()
    subtype_class_weights = torch.tensor(weights_ens, dtype=torch.float32, device=DEVICE)

    print(f"[FOLD {fold_idx}] Subtype |"
          f"subtype_class_weights={weights_ens.round(4).tolist()} ")

    return train_loader, val_loader, subtype_class_weights

In [9]:
# 6. metrics

def _compute_binary_metrics(labels: list, probs: list,
                            threshold: float = THRESHOLD) -> dict:
    """
    ACC, AUC, F1, Recall, Confusion Matrix for malignancy.
    """
    preds = (np.array(probs) >= threshold).astype(int)
    labs = np.array(labels).astype(int)
    auc = roc_auc_score(labs, probs) if len(set(labs)) > 1 else float("nan")

    return {
        "acc": float(accuracy_score(labs, preds)),
        "auc": float(auc),
        "f1": float(f1_score(labs, preds, zero_division=0)),
        "recall": float(recall_score(labs, preds, zero_division=0)),
        "cm": confusion_matrix(labs, preds).tolist(),
    }

def _compute_multiclass_metrics(true: np.ndarray, preds: np.ndarray) -> dict:
    """
    ACC, marco-F1, Confusion Matrix for subtype.
    """
    return {
        "acc": float(accuracy_score(true, preds)),
        "macro_f1": float(f1_score(true, preds, average="macro", zero_division=0)),
        "cm": confusion_matrix(true, preds).tolist(),
    }

In [10]:
# 7. Train / Eval: Malignancy
def train_malignancy_epoch(model: nn.Module, loader: DataLoader, 
                           criterion: nn.BCEWithLogitsLoss, 
                           optimizer, scaler) -> dict:
    model.train()
    running_loss = 0.0
    all_label, all_prob = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE).view(-1)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_label.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_prob.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())

    m = _compute_binary_metrics(all_label, all_prob)
    m["loss"] = running_loss / len(loader.dataset)

    return m


@torch.no_grad()
def eval_malignancy_epoch(model: nn.Module, loader: DataLoader, 
                         criterion: nn.BCEWithLogitsLoss) -> dict:
    model.eval()
    running_loss = 0.0
    all_label, all_prob = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE).view(-1)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        all_label.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_prob.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())

    m = _compute_binary_metrics(all_label, all_prob)
    m["loss"] = running_loss / len(loader.dataset)

    return m

In [11]:
# 8. Train / Eval: Subtype
def train_subtype_epoch(model: nn.Module, loader: DataLoader, 
                        criterion: FocalLoss, 
                        optimizer, scaler) -> dict:
    model.train()
    running_loss = 0.0
    all_true, all_pred = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        subtypes = batch["mal_subtype"].to(DEVICE).view(-1)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, subtypes)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_true.extend(subtypes.cpu().numpy().tolist())
        all_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    m = _compute_multiclass_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = running_loss / len(loader.dataset)
    return m


@torch.no_grad()
def eval_subtype_epoch(model: nn.Module, loader: DataLoader, 
                       criterion: FocalLoss) -> dict:
    model.eval()
    running_loss = 0.0
    all_true, all_pred = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        subtypes = batch["mal_subtype"].to(DEVICE).view(-1)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, subtypes)

        running_loss += loss.item() * imgs.size(0)
        all_true.extend(subtypes.cpu().numpy().tolist())
        all_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    m = _compute_multiclass_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = running_loss / len(loader.dataset)

    return m

In [12]:
# 9. Cascade Evaludation

@torch.no_grad()
def evaluate_cascade(mal_model: nn.Module, sub_model: nn.Module,
                     loader: DataLoader) -> dict:
    """
    실제 cascade 추론:
        1단계: malignancy 예측
        2단계: malignant으로 예측된 샘플만 subtype 예측
    분모: GT 악성 전체
    """
    mal_model.eval()
    sub_model.eval()

    all_label, all_mal_prob, all_mal_pred = [], [], []
    all_subtype_true, all_subtype_pred = [], []
    all_mal_mask = []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].cpu().numpy().astype(int).ravel()
        mal_mask = batch["mal_mask"].cpu().numpy().astype(bool).ravel()
        sub_true = batch["mal_subtype"].cpu().numpy().astype(int).ravel()
        

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            mal_logits = mal_model(imgs)
            sub_logits = sub_model(imgs)

        mal_prob = torch.sigmoid(mal_logits).cpu().numpy().ravel()
        mal_pred = (mal_prob >= TRHESHOLD).astype(int)
        sub_pred = sub_logits.argmax(dim=1).cpu().numpy().astype(int).ravel()

        all_label.extend(labels.tolist())
        all_mal_prob.extend(mal_prob.tolist())
        all_mal_pred.extend(mal_pred.toslit())
        all_mal_mask.extend(mal_mask.tolist())
        all_subtype_true.extend(sub_true.tolist())
        all_subtype_pred.extend(sub_pred.tolist())

    all_label = np.array(all_label).astype(int)
    all_mal_pred = np.array(all_mal_pred).astype(int)
    all_mal_mask = np.array(all_mal_mask).astype(bool)
    all_sub_true = np.array(all_subtype_true).astype(int)
    all_sub_pred = np.array(all_subtype_pred).astype(int)

    # Malignancy metrics
    mal_m = _compute_binary_metrics(all_label.toslit(), all_mal_prob)

    # Oracle subtype (GT 악성 기준, 1단계 무시)
    oracle_true = all_sub_true[all_mal_mask]
    oracle_pred = all_sub_pred[all_mal_mask]
    oracle_m = _compute_multiclass_metrics(oracel_true, oracle_pred)

    # Cascade subtype (실제 cascade 기준)
    gt_mal_mask = (all_label == 1)
    if get_mal_mask.sum() > 0:
        gate_recall = (all_mal_pred[gt_mal_mask] == 1).mean()
        tp_mask = (all_mal_pred[gt_mal_mask] == 1)
        tp_sub_true = oracle_true[tp_mask]
        tp_sub_pred = oracle_pred[tp_mask]
        cascade_acc = (
            (tp_sub_true == tp_sub_pred).sum() / gt_mal_mask.sum()
            if tp_mask.sum() > 0 else 0.0
        )
    else:
        gate_recall = cascade_acc = np.nan

    print(f"  [Cascade Eval]"
          f"  mal_auc={mal_m['auc']:.4f}"
          f"  mal_f1={mal_m['f1']:.4f}"
          f"  oracle_acc={oracle_m['acc']:.4f}"
          f"  oracle_f1={oracle_m['f1']:.4f}"
          f"  cascade_acc={float(cascade_acc):.4f}"
          f"  gate_recall={float(gate_recall):.4f}")

    return {
        "mal_auc": mal_m['auc'],
        "mal_f1": mal_m['f1'],
        "mal_recall": mal_m['recall'],
        "mal_cm": mal_m['cm'],
        "oracle_acc": oracle_m["acc"],
        "oracle_macro_f1": oracle_m["macro_f1"],
        "oracle_cm": oracle_m["cm"],
        "cascade_cm": float(cascade_acc),
        "gate_recall": float(gate_recall),
    }


In [13]:
# 10. Step 1: Malignancy 모델 학습

def train_malignancy_model(fold_idx: int) -> nn.Module:
    print(f"\n[Step 1] Malignancy | Fold {fold_idx}/{NUM_FOLDS}")

    train_loader, val_loader, pos_weight = make_malignancy_loaders(fold_idx)

    encoder, feat_dim = build_encoder(freeze_backbone=True,
                                      unfreeze_layer4=True)
    model = MalignancyModel(encoder, feat_dim).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in model.named_parameters()
                       if p.requires_grad and "encoder" in n], "lr": MAL_LR_BACKBONE},
         {"params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "encoder" not in n], "lr": MAL_LR_HEAD}],
        weight_decay=MAL_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAL_EPOCHS, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_auc, best_state, patience_counter = 0.0, None, 0

    for epoch in range(1, MAL_EPOCHS + 1):
        tr = train_malignancy_epoch(model, train_loader, criterion,
                                    optimizer, scaler)
        vl = eval_malignancy_epoch(model, val_loader, criterion)
        scheduler.step()

        print(f"  Epoch {epoch:02d}/{MAL_EPOCHS} "
              f"loss={tr['loss']:.4f}|{vl['loss']:.4f} "
              f"auc={tr['auc']:.4f}|{vl['auc']:.4f} "
              f"f1={tr['f1']:.4f}|{vl['f1']:.4f} "
              f"patience={patience_counter}|{MAL_PATIENCE} ")

        if vl["auc"] > best_auc + MAL_MIN_DELTA:
            best_auc = vl["auc"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= MAL_PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    torch.save(best_state,
               OUTPUT_ROOT / f"malignancy_fold{fold_idx}_best.pth")
    print(f"  Best val AUC: {best_auc:.4f}")
    return model


In [14]:
# 11. Step 2: Subtype 모델 학습

def train_subtype_model(fold_idx: int) -> nn.Module:
    print(f"\n[Step 2] Subtype | Fold {fold_idx}/{NUM_FOLDS}")

    train_loader, val_loader, subtype_weights = make_subtype_loaders(fold_idx)

    encoder, feat_dim = build_encoder(freeze_backbone=True,
                                      unfreeze_layer4=True,
                                      unfreeze_layer3=True)
    model = SubtypeModel(encoder, feat_dim).to(DEVICE)
    criterion = FocalLoss(weight=subtype_weights,
                          gamma=FOCAL_GAMMA,
                          smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in model.named_parameters()
                       if p.requires_grad and "encoder" in n], "lr": SUB_LR_BACKBONE},
         {"params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "encoder" not in n], "lr": SUB_LR_HEAD}],
        weight_decay=SUB_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAL_EPOCHS, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_f1, best_state, patience_counter = 0.0, None, 0

    for epoch in range(1, SUB_EPOCHS + 1):
        tr = train_subtype_epoch(model, train_loader, criterion,
                                    optimizer, scaler)
        vl = eval_subtype_epoch(model, val_loader, criterion)
        scheduler.step()

        print(f"  Epoch {epoch:02d}/{SUB_EPOCHS} "
              f"loss={tr['loss']:.4f}|{vl['loss']:.4f} "
              f"acc={tr['acc']:.4f}|{vl['acc']:.4f} "
              f"f1={tr['macro_f1']:.4f}|{vl['macro_f1']:.4f} "
              f"patience={patience_counter}|{SUB_PATIENCE} ")

        if vl["macro_f1"] > best_f1 + SUB_MIN_DELTA:
            best_f1 = vl["macro_f1"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= SUB_PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    torch.save(best_state,
               OUTPUT_ROOT / f"subtype_fold{fold_idx}_best.pth")
    print(f"  Best val F1: {best_f1:.4f}")
    return model


In [15]:
#12. Cascade 평가

def load_malignancy_model(fold_idx: int) -> nn.Module:
    encoder, feat_dim = build_encoder(freeze_backbone=True)
    model = MalignancyModel(encoder, feat_dim).to(DEVICE)
    state = torch.load(OUTPUT_ROOT / f"malignancy_fold{fold_idx}_best.pth",
                       map_location=DEVICE)
    model.load_state_dict(state)
    return model

def load_subtype_model(fold_idx: int) -> nn.Module:
    encoder, feat_dim = build_encoder(freeze_backbone=True)
    model = SubtypeModel(encoder, feat_dim).to(DEVICE)
    state = torch.load(OUTPUT_ROOT / f"subtype_fold{fold_idx}_best.pth",
                       map_location=DEVICE)
    model.load_state_dict(state)
    return model

In [16]:
# 13. Full CV experiment

def run_cv_experiment():
    print(f"\n{'='*60}")
    print(f"  Device: {DEVICE}")
    print(f"{'='*60}")
    
    fold_results = []

    for fold_idx in range(1, NUM_FOLDS + 1):
        print(f"\n{'-'*50}")
        print(f" FOLD {fold_idx}/{NUM_FOLDS}")
        print(f"{'-'*50}")
        
        # Step 1: Malignancy 모델 학습
        mal_model = train_malignancy_model(fold_idx)

        # Step 2: Subtype 모델 학습
        sub_model = train_subtype_model(fold_idx)

        # Step 3: Cascade 평가 (val set 기준)
        print(f"\n[Step 3] Cascade Evaluation | Fold {fold_idx}")
        _, val_loader, _ = mak_malignancy_loaders(fold_idx)
        result = evaluate_cascade(mal_model, sub_model, val_loader)
        fold_results.append(result)

    # CV 결과 요약
    print(f"\n{'='*60}")
    print(f"  5-Fold CV Summary")
    print(f"{'='*60}")

    for key in ["mal_auc", "mal_f1", "oracle_acc", "oracle_macro_f1",
                "cascade_acc", "gate_recall"]:
        vals = [r[key] for r in fold_results
                if not np.isnan(r[key])]
        print(f"  {key:25s}: {np.mean(vals):.4f} +- {np.std(vals):.4f}")

    return fold_results

In [17]:
# 14. Entry point

#if __name__ == "__main__":
#    results = run_cv_experiment()

In [18]:
for fold_idx in range(1, NUM_FOLDS + 1):
    print(f"\n{'-'*50}")
    print(f" FOLD {fold_idx}/{NUM_FOLDS}")
    print(f"{'-'*50}")
        
    # Step 2: Subtype 모델 학습
    sub_model = train_subtype_model(fold_idx)


--------------------------------------------------
 FOLD 1/5
--------------------------------------------------

[Step 2] Subtype | Fold 1/5
  [MalignantOnly] 총 711명 |subtype 분포: [160, 110, 67, 125, 249]
  [MalignantOnly] 총 98명 |subtype 분포: [8, 5, 17, 6, 62]
[FOLD 1] Subtype |subtype_class_weights=[1.5494, 2.248, 3.6828, 1.9797, 1.0] 
  [RadImageNet] loaded RadImageNet_resnet18.pth
   missing keys (2): ['fc.weight', 'fc.bias']
  Epoch 01/80 loss=2.2525|2.0727 acc=0.1744|0.1531 f1=0.1378|0.0984 patience=0|20 
  Epoch 02/80 loss=2.2331|2.0829 acc=0.2110|0.1429 f1=0.1910|0.0941 patience=0|20 
  Epoch 03/80 loss=2.1830|2.0789 acc=0.2588|0.1531 f1=0.2554|0.0978 patience=1|20 
  Epoch 04/80 loss=2.1674|2.0709 acc=0.2996|0.1735 f1=0.2928|0.1201 patience=2|20 
  Epoch 05/80 loss=2.1290|2.0633 acc=0.3643|0.2347 f1=0.3605|0.1992 patience=0|20 
  Epoch 06/80 loss=2.0829|2.0600 acc=0.3474|0.3265 f1=0.3409|0.2470 patience=0|20 
  Epoch 07/80 loss=2.0551|2.0530 acc=0.3629|0.3061 f1=0.3588|0.2012 pa

KeyboardInterrupt: 